# Bollywood Film & Song Scraper

Scrapes every Hindi-language film listed on Wikipedia (1931–present) and extracts structured soundtrack data — song titles, playback singers, and music directors — into a single CSV.

## How it works

The scraper walks Wikipedia's yearly index pages ("List of Hindi films of 20XX"), follows each film's link, locates the soundtrack table, and pulls out per-song metadata. It handles the wide variation in how Wikipedia editors format these tables across nine decades of films:

- **Rowspan/colspan resolution** builds a normalized 2-D grid from each table so column indices stay correct even when a music director cell spans an entire album
- **Two-pass column detection** claims lyricist, singer, composer, and duration columns first, then identifies the song title from whatever remains — preventing lyrics columns from being misidentified as titles
- **`<th scope="row">` awareness** picks up song titles that Wikipedia puts in header cells rather than data cells, a common pattern in modern soundtrack tables
- **Three fallback strategies** try structured table extraction first, then bulleted lists under "Soundtrack" or "Songs" headings

## Filtering and cleanup

- Skips non-film pages (bios, plays, disambiguation) using infobox and category heuristics
- Excludes navbox/sidebar/infobox tables, non-Hindi language sections, and discography/awards/controversy tables
- Caps song tables at 40 rows to avoid scraping filmography lists disguised as tracklists
- Strips citation brackets, a.k.a. suffixes, smart quotes, and concatenated composer–lyricist names

## Performance

- Concurrent film-page fetching within each year (configurable thread pool)
- Connection reuse via `requests.Session()`
- Resume support: detects years already in the output CSV and skips them
- Tunable per-request and per-year sleep intervals for polite crawling

## Output

`bollywood_full_history.csv` with columns: **Year, Movie, Song, Singer, Composer**.

In [11]:
import os
import re
import csv
import time
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

BASE_URL = "https://en.wikipedia.org"
CATEGORY_URL = (
    "https://en.wikipedia.org/wiki/"
    "Category:Lists_of_Hindi_films_by_year"
)
HEADERS = {
    "User-Agent": (
        "BollywoodSongScraper/2.0 "
        "(academic research; polite crawl; contact: you@example.com)"
    )
}

OUTPUT_FILE = "bollywood_full_history.csv"
MAX_WORKERS = 5          # concurrent film-page fetches per year
SLEEP_PER_FILM = 0.3     # seconds between film requests (per thread)
SLEEP_PER_YEAR = 0.5     # seconds between year pages

# Reusable session — keeps TCP connections alive across requests
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

# Thread-safe CSV writer
CSV_LOCK = Lock()

# Headings that signal a table we should skip entirely
SKIP_HEADINGS = [
    "discography", "filmography", "samples", "interpolation",
    "plagiarism", "controversy", "accolades", "awards",
    "career", "personal life", "bibliography", "other works",
]

# Non-Hindi language sections to skip
NON_HINDI_LANGS = [
    "gujarati", "tamil", "telugu", "marathi",
    "kannada", "bengali", "malayalam", "punjabi",
    "bhojpuri", "assamese", "odia", "urdu",
]

# Words that disqualify a cell from being a song title
EXCLUDE_WORDS = [
    "portal", "imdb", "external links", "references", "bibliography",
    "see also", "citation", "awards", "box office", "cast", "crew",
    "reception", "production", "plot", "release", "filmography",
    "accolades", "discography", "career", "personal life",
]

In [ ]:
# ── Issue 2: rewritten page_is_film ──────────────────────────────────────────
def page_is_film(soup):
    """Return True if the page describes a specific film.

    Strategy (priority order):
      1. Infobox contains unambiguous FILM fields → True immediately.
      2. Infobox contains unambiguous NON-FILM fields (bio / play) → False.
      3. Page categories contain film-year pattern → True.
      4. Default → False.

    Key fix vs original:
    - "genre" and "original language" are NO LONGER in the negative list;
      they appear in film infoboxes and caused the 73–85 % false-negative rate.
    - "directed by" is sufficient on its own for a positive match.
    """
    infobox = soup.find("table", class_="infobox")
    if infobox:
        itext = infobox.get_text(" ", strip=True).lower()

        FILM_POSITIVE = [
            "directed by", "produced by", "screenplay by",
            "release date", "music by", "starring",
            "cinematography", "distributed by",
        ]
        if any(kw in itext for kw in FILM_POSITIVE):
            return True

        NON_FILM_NEGATIVE = [
            "date premiered",
            "born",
            "died",
            "occupation",
            "nationality",
            "alma mater",
            "years active",
            "known for",
            "spouse",
        ]
        if any(kw in itext for kw in NON_FILM_NEGATIVE):
            return False

    cats = soup.find("div", id="mw-normal-catlinks")
    if cats:
        for a in cats.find_all("a"):
            t = a.get_text().lower()
            if re.search(r'\d{4}.*films?', t) or "language film" in t:
                return True
            if "hindi film" in t or "bollywood film" in t:
                return True

    return False

In [13]:
def identify_columns(header_row):
    """Map header cells to column roles: title, singer, composer, lyricist, duration.

    Returns {role: column_index}. Roles not found are omitted.

    Two-pass approach:
      Pass 1 — claim specific non-title roles (lyricist, singer, composer, duration)
               so they can never be mistaken for the title column.
      Pass 2 — claim title from whatever remains.
      Pass 3 — post-hoc inference if title is still missing.
    """
    cells = header_row.find_all(["th", "td"])
    headers = [c.get_text(strip=True).lower() for c in cells]
    col_map = {}
    claimed = set()

    # Track number columns — always skip
    skip_indices = set()
    for i, h in enumerate(headers):
        if h in ("#", "no.", "no", "sr.", "sr", "s.no", "s.no.", "s. no.",
                 "s no", "sl.", "sl.no", "sl. no.", "sl no"):
            skip_indices.add(i)

    # ---- Pass 1: claim non-title roles first ----
    for i, h in enumerate(headers):
        if i in skip_indices:
            continue

        # Duration
        if not col_map.get("duration"):
            if any(kw in h for kw in ["duration", "length"]):
                col_map["duration"] = i
                claimed.add(i)
                continue
            if h == "time":
                col_map["duration"] = i
                claimed.add(i)
                continue

        # Lyricist — MUST be checked before title so "lyrics" is never
        # mistaken for a title keyword
        if not col_map.get("lyricist"):
            if any(kw in h for kw in ["lyric", "poet", "penned by"]):
                col_map["lyricist"] = i
                claimed.add(i)
                continue
            if ("writer" in h or "written by" in h) and "song" not in h:
                col_map["lyricist"] = i
                claimed.add(i)
                continue

        # Singer
        if not col_map.get("singer"):
            if any(kw in h for kw in ["singer", "artist", "sung by",
                                       "playback", "vocalist", "performer",
                                       "voice", "rendered by"]):
                col_map["singer"] = i
                claimed.add(i)
                continue

        # Composer
        if not col_map.get("composer"):
            if any(kw in h for kw in ["music", "composer", "composed by",
                                       "music director", "music by"]):
                col_map["composer"] = i
                claimed.add(i)
                continue

    # ---- Pass 2: claim title from remaining columns ----
    for i, h in enumerate(headers):
        if i in skip_indices or i in claimed:
            continue
        if not col_map.get("title"):
            if any(kw in h for kw in ["song", "title", "track"]):
                # Guard: don't match headers like "singer name" or "artist name"
                if not any(kw in h for kw in ["singer", "artist", "playback"]):
                    col_map["title"] = i
                    claimed.add(i)
                    break

    # ---- Pass 3: post-hoc inference ----
    if "title" not in col_map:
        anchor = col_map.get("singer", col_map.get("composer", len(headers)))
        for i in range(anchor):
            if i not in claimed and i not in skip_indices:
                col_map["title"] = i
                claimed.add(i)
                break

    # Last resort: first unclaimed column anywhere
    if "title" not in col_map:
        for i in range(len(headers)):
            if i not in claimed and i not in skip_indices:
                col_map["title"] = i
                claimed.add(i)
                break

    # Infer singer from first unclaimed column after title
    if "title" in col_map and "singer" not in col_map:
        title_idx = col_map["title"]
        for i in range(title_idx + 1, len(headers)):
            if i not in claimed and i not in skip_indices:
                h = headers[i]
                if not any(kw in h for kw in ["duration", "length", "time",
                                               "music", "composer"]):
                    col_map["singer"] = i
                    claimed.add(i)
                    break

    return col_map

In [ ]:
def get_songs_from_film(film_url, film_title):
    """Scrape songs from a single film's Wikipedia page.

    Returns (songs_list, was_film) where:
      songs_list  — list of song dicts (may be empty)
      was_film    — True if page_is_film() passed, False if rejected
    """
    songs = []
    page_composer = ""

    try:
        res = SESSION.get(film_url, timeout=10)
        soup = BeautifulSoup(res.text, 'html.parser')

        if not page_is_film(soup):
            print(f"  Skipping {film_title}: not a film page")
            return [], False

        infobox = soup.find("table", class_="infobox")
        if infobox:
            for row in infobox.find_all("tr"):
                th = row.find("th")
                td = row.find("td")
                if th and td and "music" in th.get_text().lower():
                    page_composer = clean_cell(td.get_text(strip=True))
                    page_composer = split_composer_lyricist(page_composer)
                    break

        all_tables = soup.find_all("table")
        for table in all_tables:
            if table_has_skip_class(table):
                continue
            if has_navbox_ancestor(table):
                continue

            prev_heading = table.find_previous(["h2", "h3", "h4"])
            if heading_is_skippable(prev_heading):
                continue

            table_text = table.get_text().lower()
            if not ("song" in table_text or "track" in table_text
                    or "singer" in table_text or "playback" in table_text):
                continue

            rows = table.find_all("tr")
            if not rows:
                continue

            grid = _resolve_rowspans(table)
            if len(grid) < 2:
                continue

            if len(grid) > 41:
                continue

            clean_headers = []
            seen_cells = set()
            for idx, cell in enumerate(grid[0]):
                cell_id = id(cell) if cell else idx
                if cell_id not in seen_cells:
                    seen_cells.add(cell_id)
                    clean_headers.append(
                        cell.get_text(strip=True).lower() if cell else ""
                    )
                else:
                    clean_headers.append("")

            class FakeRow:
                def __init__(self, texts):
                    self._texts = texts
                def find_all(self, tags):
                    class FakeCell:
                        def __init__(self, text):
                            self._text = text
                        def get_text(self, strip=False):
                            return self._text.strip() if strip else self._text
                    return [FakeCell(t) for t in self._texts]

            col_map = identify_columns(FakeRow(clean_headers))

            if "title" in col_map and len(grid) > 1:
                sample_cell = (grid[1][col_map["title"]]
                               if col_map["title"] < len(grid[1]) else None)
                if sample_cell is not None:
                    sample = sample_cell.get_text(strip=True)
                    if is_duration(sample):
                        del col_map["title"]

            for row_cells in grid[1:]:
                has_td = any(
                    c is not None and c.name == "td" for c in row_cells
                )
                has_th_only = all(
                    c is None or c.name == "th" for c in row_cells
                )
                if has_th_only and not has_td:
                    continue

                n_cols = len(row_cells)

                if "title" in col_map and col_map["title"] < n_cols:
                    title_cell = row_cells[col_map["title"]]
                    if title_cell is None:
                        continue
                    title_text = clean_song(title_cell.get_text(strip=True))
                    if not song_title_looks_valid(title_text):
                        continue

                    singer = ""
                    composer = ""
                    if "singer" in col_map and col_map["singer"] < n_cols:
                        sc = row_cells[col_map["singer"]]
                        if sc is not None:
                            raw = sc.get_text(strip=True)
                            if not is_duration(raw):
                                singer = clean_cell(raw)
                    if "composer" in col_map and col_map["composer"] < n_cols:
                        cc = row_cells[col_map["composer"]]
                        if cc is not None:
                            raw = cc.get_text(strip=True)
                            if not is_duration(raw):
                                composer = split_composer_lyricist(
                                    clean_cell(raw))

                    songs.append({
                        "title": title_text,
                        "singer": singer,
                        "composer": composer or page_composer,
                    })
                else:
                    for cell in row_cells:
                        if cell is None:
                            continue
                        text = clean_song(cell.get_text(strip=True))
                        if song_title_looks_valid(text):
                            songs.append({"title": text, "singer": "",
                                          "composer": page_composer})
                            break

        if not songs:
            songs = _extract_list_songs(soup, "soundtrack", page_composer)
        if not songs:
            songs = _extract_list_songs(soup, "song", page_composer)

    except Exception as e:
        print(f"Error on {film_title}: {e}")
        return [], False

    seen = set()
    unique = []
    for s in songs:
        key = s["title"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(s)

    return unique, True

In [ ]:
def _link_looks_like_person(tag_a):
    """Return True if an <a> tag likely points to a person's biography page."""
    if tag_a is None:
        return False
    href = tag_a.get("href", "")
    text = tag_a.get_text(strip=True).lower()

    person_keywords = [
        "director", "producer", "writer", "actor", "actress",
        "cinematographer", "editor", "composer", "lyricist",
    ]
    if any(kw in text for kw in person_keywords):
        return True

    if re.match(r'^/wiki/[A-Z][a-z]+(?:_[A-Z][a-z]+)+$', href):
        if not re.search(r'[\d()]', href):
            return True

    return False


def _score_table_as_film_list(table, headers_text):
    """Return a score for how likely this table is the year's film index."""
    score = 0
    classes = " ".join(table.get("class", []))
    if "wikitable" in classes:
        score += 10
    if "sortable" in classes:
        score += 3

    htext = " ".join(headers_text).lower()
    for kw in ["film", "title", "director", "cast", "release"]:
        if kw in htext:
            score += 5

    n_cols = len(headers_text)
    if 3 <= n_cols <= 8:
        score += 3

    n_rows = len(table.find_all("tr"))
    if n_rows >= 10:
        score += 2
    if n_rows >= 50:
        score += 3

    return score


def scrape_year_page(url, year):
    """Scrape all films and songs from a 'List of Hindi films of XXXX' page."""
    all_data = []
    res = SESSION.get(url, timeout=15)
    soup = BeautifulSoup(res.text, 'html.parser')

    candidate_tables = []
    for table in soup.find_all("table"):
        if table_has_skip_class(table):
            continue
        if has_navbox_ancestor(table):
            continue
        rows = table.find_all("tr")
        if len(rows) < 3:
            continue
        header_cells = rows[0].find_all(["th", "td"])
        headers_text = [h.get_text(strip=True) for h in header_cells]
        score = _score_table_as_film_list(table, headers_text)
        if score >= 5:
            candidate_tables.append((score, table, headers_text))

    candidate_tables.sort(key=lambda x: x[0], reverse=True)
    candidate_tables = candidate_tables[:5]

    film_jobs = []
    seen_urls = set()

    for score, table, first_headers in candidate_tables:
        rows = table.find_all("tr")
        if not rows:
            continue

        headers_lower = [h.lower() for h in first_headers]
        try:
            title_idx = next(
                i for i, h in enumerate(headers_lower)
                if 'title' in h or 'film' in h
            )
        except StopIteration:
            title_idx = 0

        for row in rows[1:]:
            cols = row.find_all(["td", "th"])
            if not cols:
                continue

            if title_idx < len(cols):
                target_cell = cols[title_idx]
            else:
                continue

            link = target_cell.find("a")

            if link is None and title_idx + 1 < len(cols):
                link = cols[title_idx + 1].find("a")
                if link:
                    target_cell = cols[title_idx + 1]

            if link is None:
                continue

            if _link_looks_like_person(link):
                continue

            film_title = clean_title(link.get_text(strip=True))
            if not film_title:
                continue

            if "silent" in row.get_text().lower():
                continue

            href = link.get("href", "")
            if not href or href.startswith("#"):
                continue

            film_url = BASE_URL + href
            if film_url in seen_urls:
                continue
            seen_urls.add(film_url)
            film_jobs.append((film_title, film_url))

    if not film_jobs:
        print(f"  [{year}] WARNING: no film links found — page may use "
              f"an unusual layout.")

    def _fetch_one(args):
        title, furl = args
        time.sleep(SLEEP_PER_FILM)
        print(f"[{year}] Scraping: {title}")
        songs, was_film = get_songs_from_film(furl, title)
        return title, furl, songs, was_film

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(_fetch_one, job): job for job in film_jobs}
        for future in as_completed(futures):
            title, furl, songs, was_film = future.result()
            if songs:
                for s in songs:
                    all_data.append([
                        year, title, s["title"], s["singer"], s["composer"]
                    ])
            elif was_film:
                _log_no_songs(year, title, furl)

    return all_data

In [16]:
def get_yearly_list_urls():
    """Find all 'List of Hindi/Bollywood films of XXXX' URLs from the category page.

    Returns [{year, url}, ...] sorted ascending, starting from 1931.
    """
    res = SESSION.get(CATEGORY_URL, timeout=15)
    soup = BeautifulSoup(res.text, 'html.parser')

    year_links = []
    for a in soup.select("#mw-pages a"):
        title = a.get_text()
        if re.search(r"List of (Hindi|Bollywood) films of \d{4}", title):
            year_match = re.search(r"\d{4}", title)
            yr = int(year_match.group())
            if yr >= 1931:
                year_links.append({
                    "year": yr,
                    "url": BASE_URL + a['href'],
                })

    return sorted(year_links, key=lambda x: x['year'])

In [17]:
def already_scraped_years(filepath):
    """Return the set of years already present in the output CSV."""
    years = set()
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if row:
                    try:
                        years.add(int(row[0]))
                    except ValueError:
                        pass
    except FileNotFoundError:
        pass
    return years

In [ ]:
_NO_SONGS_LOCK = Lock()

def _log_no_songs(year, title, url):
    """Append a row to the no-songs audit log."""
    with _NO_SONGS_LOCK:
        write_header = not os.path.exists(NO_SONGS_LOG)
        with open(NO_SONGS_LOG, 'a', newline='', encoding='utf-8') as f:
            w = csv.writer(f)
            if write_header:
                w.writerow(["Year", "Movie", "URL"])
            w.writerow([year, title, url])


def master_scraper():
    """Scrape every year from 1931 onwards."""
    year_indices = get_yearly_list_urls()
    done_years = already_scraped_years(OUTPUT_FILE)
    if done_years:
        print(f"Resuming — skipping {len(done_years)} already-scraped years.")

    if not os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f).writerow(['Year', 'Movie', 'Song', 'Singer', 'Composer'])

    for entry in year_indices:
        yr = entry['year']
        if yr in done_years:
            print(f"[{yr}] Already scraped — skipping.")
            continue

        print(f"\n--- PROCESSING YEAR: {yr} ---")
        yearly_data = scrape_year_page(entry['url'], yr)

        if yearly_data:
            with CSV_LOCK:
                with open(OUTPUT_FILE, 'a', newline='', encoding='utf-8') as f:
                    csv.writer(f).writerows(yearly_data)
            print(f"Saved {len(yearly_data)} tracks for {yr}.")
        else:
            print(f"No tracks found for {yr}.")

        time.sleep(SLEEP_PER_YEAR)


if __name__ == "__main__":
    master_scraper()

In [19]:
if __name__ == "__main__":
    master_scraper()


--- PROCESSING YEAR: 1931 ---
[1931] Scraping: Abhishek
[1931] Scraping: Alam Ara
[1931] Scraping: Aatishe Ishq
[1931] Scraping: Abul Hasan
[1931] Scraping: Afghan Abla
[1931] Scraping: Albeli Mumbai
  Skipping Albeli Mumbai: not a film page
  Skipping Abul Hasan: not a film page
  Skipping Afghan Abla: not a film page
  Skipping Abhishek: not a film page
  Skipping Aatishe Ishq: not a film page
[1931] Scraping: Alladin and the Wonderful Lamp
[1931] Scraping: Amir Khan
[1931] Scraping: Aparadhi
[1931] Scraping: Awara Raqasa
[1931] Scraping: Anang Sena
  Skipping Alladin and the Wonderful Lamp: not a film page
  Skipping Amir Khan: not a film page
  Skipping Aparadhi: not a film page
  Skipping Awara Raqasa: not a film page
  Skipping Anang Sena: not a film page
[1931] Scraping: Baaz Bahadur
[1931] Scraping: Badmash
[1931] Scraping: Bahadur Beti
[1931] Scraping: Banke Sawariya
[1931] Scraping: Bewafa Ashiq
  Skipping Badmash: not a film page
  Skipping Baaz Bahadur: not a film page
  S